In [2]:
import os
import cv2
import numpy as np
import glob
from tqdm import tqdm

input_dir = r"cleaned_dataset\mask_rotated"
output_dir = r"cleaned_dataset\mask_resized"
target_size = 512

os.makedirs(output_dir, exist_ok=True)

img_paths = glob.glob(os.path.join(input_dir, "*.jpg"))
total_found = len(img_paths)
print(f"Found {total_found} images")

processed = 0
skipped = 0
failed = 0

for img_path in tqdm(img_paths, unit="it"):
    filename = os.path.basename(img_path)
    output_path = os.path.join(output_dir, filename)

    if os.path.exists(output_path):
        skipped += 1
        continue

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        failed += 1
        continue

    try:
        h, w = img.shape
        scale = target_size / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        
        resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        
        canvas = np.zeros((target_size, target_size), dtype=np.uint8)
        y_off = (target_size - new_h) // 2
        x_off = (target_size - new_w) // 2
        
        canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
        cv2.imwrite(output_path, canvas)
        processed += 1
    except:
        failed += 1

print("\nDone!")
print(f"Processed: {processed}")
print(f"Skipped: {skipped}")
print(f"Failed: {failed}")
print(f"Total: {total_found}")

Found 4588 images


100%|██████████| 4588/4588 [00:00<00:00, 34475.75it/s]


Done!
Processed: 0
Skipped: 4588
Failed: 0
Total: 4588


In [4]:
import os
import cv2
import numpy as np
import glob
from tqdm import tqdm

# Configuration
input_dir = r"cleaned_dataset\mask_resized"
output_dir = r"cleaned_dataset\mask_padded"
intermediate_size = 600
final_size = 512

os.makedirs(output_dir, exist_ok=True)

img_paths = glob.glob(os.path.join(input_dir, "*.jpg"))
print(f"Found {len(img_paths)} images for padding and rescaling")

processed = 0
failed = 0

for img_path in tqdm(img_paths, unit="it"):
    filename = os.path.basename(img_path)
    output_path = os.path.join(output_dir, filename)

    # Load the 512x512 mask
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        failed += 1
        continue

    try:
        # 1. Create a 600x600 black canvas
        canvas = np.zeros((intermediate_size, intermediate_size), dtype=np.uint8)
        
        # 2. Position the OG image at top-center
        # Vertical: Top starts at 0
        # Horizontal: Center is (600 - 512) // 2 = 44
        y_off = 0
        x_off = (intermediate_size - final_size) // 2
        
        canvas[y_off:y_off+final_size, x_off:x_off+final_size] = img
        
        # 3. Rescale back to 512x512
        # Using INTER_NEAREST to preserve mask binary integrity
        final_img = cv2.resize(canvas, (final_size, final_size), interpolation=cv2.INTER_NEAREST)
        
        cv2.imwrite(output_path, final_img)
        processed += 1
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        failed += 1

print(f"\nTask Complete!")
print(f"Successfully processed: {processed}")
print(f"Failed: {failed}")

Found 4588 images for padding and rescaling


100%|██████████| 4588/4588 [00:10<00:00, 448.12it/s]


Task Complete!
Successfully processed: 4588
Failed: 0


In [6]:
import pandas as pd
import os
import shutil
from tqdm import tqdm

# 1. Configuration - Define your paths here
INPUT_CSV = 'cleaned_dataset/new_csv/filtered_metadata.csv'
SOURCE_MASK_DIR = 'cleaned_dataset/mask_padded'

TRAIN_CSV_PATH = 'cleaned_dataset/new_csv/train_metadata.csv'
INF_CSV_PATH = 'cleaned_dataset/new_csv/inference_metadata.csv'

TRAIN_MASK_DIR = 'cleaned_dataset/split/train_masks'
INF_MASK_DIR = 'cleaned_dataset/split/inference_masks'

# 2. Create Target Directories
os.makedirs(TRAIN_MASK_DIR, exist_ok=True)
os.makedirs(INF_MASK_DIR, exist_ok=True)

# 3. Load and Split Metadata
df = pd.read_csv(INPUT_CSV)

# Split: Train has both length and width; Inference is missing at least one
train_df = df.dropna(subset=['length_mm', 'width_mm'])
inf_df = df[df['length_mm'].isna() | df['width_mm'].isna()]

# Save the updated CSVs
train_df.to_csv(TRAIN_CSV_PATH, index=False)
inf_df.to_csv(INF_CSV_PATH, index=False)

print(f"Metadata Processing Complete.")
print(f"-> Found {len(train_df)} training samples and {len(inf_df)} inference samples.")

# 4. Helper Function with Skip Logic and Progress Bar
def organize_files(dataframe, target_dir, label="Processing"):
    stats = {"copied": 0, "skipped": 0, "missing": 0}
    
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=label):
        # Extract filename from the image_path column
        filename = os.path.basename(row['image_path'])
        src = os.path.join(SOURCE_MASK_DIR, filename)
        dst = os.path.join(target_dir, filename)
        
        # Skip logic
        if os.path.exists(dst):
            stats["skipped"] += 1
            continue
            
        # Copy logic
        if os.path.exists(src):
            shutil.copy(src, dst)
            stats["copied"] += 1
        else:
            stats["missing"] += 1
            
    return stats

# 5. Execute File Organization
print("\nOrganizing Training Masks...")
train_stats = organize_files(train_df, TRAIN_MASK_DIR, label="Train Masks")

print("\nOrganizing Inference Masks...")
inf_stats = organize_files(inf_df, INF_MASK_DIR, label="Inference Masks")

# 6. Final Summary Report
print("\n" + "="*40)
print("FINAL EXECUTION SUMMARY")
print("="*40)
print(f"TRAIN: {train_stats['copied']} copied, {train_stats['skipped']} skipped, {train_stats['missing']} missing.")
print(f"INF:   {inf_stats['copied']} copied, {inf_stats['skipped']} skipped, {inf_stats['missing']} missing.")
print("="*40)

Metadata Processing Complete.
-> Found 204 training samples and 4384 inference samples.

Organizing Training Masks...


Train Masks: 100%|██████████| 204/204 [00:02<00:00, 95.87it/s] 



Organizing Inference Masks...


Inference Masks: 100%|██████████| 4384/4384 [00:53<00:00, 81.50it/s] 


FINAL EXECUTION SUMMARY
TRAIN: 204 copied, 0 skipped, 0 missing.
INF:   4384 copied, 0 skipped, 0 missing.


In [7]:
import pandas as pd
import cv2
import numpy as np
import os
from tqdm import tqdm

TRAIN_MASK_DIR = 'cleaned_dataset/split/train_masks'
INF_MASK_DIR = 'cleaned_dataset/split/inference_masks'
TRAIN_PLOT_DIR = 'cleaned_dataset/cutoff_plot/train'
INF_PLOT_DIR = 'cleaned_dataset/cutoff_plot/inference'

INPUT_TRAIN_CSV = 'cleaned_dataset/new_csv/train_metadata.csv'
INPUT_INF_CSV = 'cleaned_dataset/new_csv/inference_metadata.csv'
OUTPUT_TRAIN_CSV = 'cleaned_dataset/new_csv/train_metadata_with_cutoffs.csv'
OUTPUT_INF_CSV = 'cleaned_dataset/new_csv/inference_metadata_with_cutoffs.csv'

import numpy as np

def find_inflection_cutoff(half_contour, is_left=True):
    pts = half_contour[np.argsort(half_contour[:, 1])]
    y_min, y_max = pts[0, 1], pts[-1, 1]
    y_start = y_min + 0.80 * (y_max - y_min)
    target_pts = pts[pts[:, 1] > y_start]

    if len(target_pts) < 12:
        return pts[-1]

    # --- 1. Original Slope Logic ---
    window = 10
    cutoff_pt = target_pts[-1]
    cutoff_idx = len(target_pts) - 1 # Keep track of the index for the refinement
    
    for i in range(len(target_pts) - window):
        p_curr = target_pts[i]
        p_future = target_pts[i + window]
        dx = p_future[0] - p_curr[0]
        dy = p_future[1] - p_curr[1]
        
        if dy <= 0:
            continue
            
        slope_inv = dx / dy
        
        if is_left:
            if slope_inv > 1.8:
                cutoff_pt = p_curr
                cutoff_idx = i
                break
        else:
            if slope_inv < -1.8:
                cutoff_pt = p_curr
                cutoff_idx = i
                break

    # --- 2. New Refinement: Inward Bulge Check ---
    # If the cutoff is too close to the top, we can't reliably check for a bulge
    if cutoff_idx < 5: 
        return cutoff_pt
        
    p_a = target_pts[0]      # Top of search area
    p_b = cutoff_pt          # Original found cutoff
    
    dx_ab = p_b[0] - p_a[0]
    dy_ab = p_b[1] - p_a[1]
    norm_ab = np.sqrt(dx_ab**2 + dy_ab**2)
    
    if norm_ab == 0:
        return cutoff_pt
        
    max_dist = -1
    best_pt = cutoff_pt
    
    # Noise handling: 
    # Buffer skips the first/last few points connected to the anchors.
    # Threshold requires a point to be visibly inside the line to count.
    buffer = 2 
    noise_threshold = 1.0 
    
    # Only check points between the top and the original cutoff
    for i in range(buffer, cutoff_idx - buffer):
        p = target_pts[i]
        
        # Calculate signed perpendicular distance using cross product
        # Z = dx_AB * dy_AP - dy_AB * dx_AP
        dx_ap = p[0] - p_a[0]
        dy_ap = p[1] - p_a[1]
        
        z = dx_ab * dy_ap - dy_ab * dx_ap
        signed_dist = z / norm_ab
        
        # In screen coords (+Y is down, +X is right):
        # A negative distance means the point is to the RIGHT of the line A->B.
        # A positive distance means the point is to the LEFT of the line A->B.
        
        if is_left:
            # Left half: looking for points on the RIGHT side of the line
            dist_inside = -signed_dist 
        else:
            # Right half: looking for points on the LEFT side of the line
            dist_inside = signed_dist
            
        # If the point is on the correct side, exceeds pixel noise, and is the furthest so far:
        if dist_inside > noise_threshold and dist_inside > max_dist:
            max_dist = dist_inside
            best_pt = p
            
    return best_pt

def process_metadata(input_csv, output_csv, mask_dir, plot_dir, desc="Processing"):
    os.makedirs(plot_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    
    cutoff_ls = []
    cutoff_rs = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        img_filename = row['image_path']
        img_path = os.path.join(mask_dir, img_filename)
        
        if not os.path.exists(img_path):
            cutoff_ls.append(None)
            cutoff_rs.append(None)
            continue

        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        
        if not contours:
            cutoff_ls.append(None)
            cutoff_rs.append(None)
            continue

        cnt = max(contours, key=cv2.contourArea).squeeze()
        x_center = (np.min(cnt[:, 0]) + np.max(cnt[:, 0])) // 2
        left_half = cnt[cnt[:, 0] < x_center]
        right_half = cnt[cnt[:, 0] >= x_center]

        l_cutoff = find_inflection_cutoff(left_half, is_left=True)
        r_cutoff = find_inflection_cutoff(right_half, is_left=False)

        cutoff_ls.append(f"{l_cutoff[0]},{l_cutoff[1]}")
        cutoff_rs.append(f"{r_cutoff[0]},{r_cutoff[1]}")

        canvas = img.copy()
        cv2.circle(canvas, tuple(l_cutoff), 2, (0, 0, 255), -1)
        cv2.circle(canvas, tuple(r_cutoff), 2, (0, 0, 255), -1)
        cv2.imwrite(os.path.join(plot_dir, img_filename), canvas)

    df['cutoff_l'] = cutoff_ls
    df['cutoff_r'] = cutoff_rs
    df['end_pt'] = np.nan
    df['curve'] = np.nan
    
    df.to_csv(output_csv, index=False)

process_metadata(INPUT_TRAIN_CSV, OUTPUT_TRAIN_CSV, TRAIN_MASK_DIR, TRAIN_PLOT_DIR, desc="Training Masks")
process_metadata(INPUT_INF_CSV, OUTPUT_INF_CSV, INF_MASK_DIR, INF_PLOT_DIR, desc="Inference Masks")

Inference Masks: 100%|██████████| 4384/4384 [00:35<00:00, 123.95it/s]


In [8]:
import pandas as pd
import cv2
import numpy as np
import os
from tqdm import tqdm

# Constants
TRAIN_MASK_DIR = 'cleaned_dataset/split/train_masks'
INF_MASK_DIR = 'cleaned_dataset/split/inference_masks'
INPUT_TRAIN_CSV = 'cleaned_dataset/new_csv/train_metadata_with_cutoffs.csv'
INPUT_INF_CSV = 'cleaned_dataset/new_csv/inference_metadata_with_cutoffs.csv'

# New Output Directories
OUTPUT_TRAIN_DIR = 'cleaned_dataset/cut/train_masks'
OUTPUT_INF_DIR = 'cleaned_dataset/cut/inference_masks'

os.makedirs(OUTPUT_TRAIN_DIR, exist_ok=True)
os.makedirs(OUTPUT_INF_DIR, exist_ok=True)

def apply_cutoff_to_image(img_path, l_pt_str, r_pt_str, save_path):
    # Load image
    img = cv2.imread(img_path)
    if img is None:
        return
    
    h, w = img.shape[:2]
    
    try:
        # Parse points from "x,y" string
        l_pt = [int(float(c)) for c in l_pt_str.split(',')]
        r_pt = [int(float(c)) for c in r_pt_str.split(',')]
    except (ValueError, AttributeError):
        # If points are NaN or malformed, just save the original or skip
        cv2.imwrite(save_path, img)
        return

    # 1. Define the line equation: y = mx + c
    # Using (x1, y1) and (x2, y2)
    x1, y1 = l_pt
    x2, y2 = r_pt
    
    if x2 - x1 != 0:
        m = (y2 - y1) / (x2 - x1)
        c = y1 - m * x1
        
        # 2. Find intersection points with the image boundaries (x=0 and x=w)
        # to ensure the cut spans the entire width
        y_at_x0 = int(m * 0 + c)
        y_at_xw = int(m * (w - 1) + c)
    else:
        # Fallback for vertical line (rare)
        y_at_x0, y_at_xw = y1, y2

    # 3. Create a polygon representing the area BELOW the line
    # Vertices: Left-intersect, Right-intersect, Bottom-Right corner, Bottom-Left corner
    cut_poly = np.array([
        [0, y_at_x0],
        [w - 1, y_at_xw],
        [w - 1, h - 1],
        [0, h - 1]
    ], dtype=np.int32)

    # 4. Fill the area below the line with Black (0)
    # We use -1 thickness to fill the polygon
    cv2.fillPoly(img, [cut_poly], (0, 0, 0))

    # Save the result
    cv2.imwrite(save_path, img)

def process_cutting(input_csv, source_dir, target_dir, desc):
    os.makedirs(target_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        img_name = row['image_path']
        l_cutoff = row['cutoff_l']
        r_cutoff = row['cutoff_r']
        
        src_path = os.path.join(source_dir, img_name)
        dst_path = os.path.join(target_dir, img_name)
        
        if os.path.exists(src_path) and pd.notna(l_cutoff) and pd.notna(r_cutoff):
            apply_cutoff_to_image(src_path, l_cutoff, r_cutoff, dst_path)
        elif os.path.exists(src_path):
            # If no cutoff found, copy original to the new folder
            img = cv2.imread(src_path)
            cv2.imwrite(dst_path, img)

# Execute
process_cutting(INPUT_TRAIN_CSV, TRAIN_MASK_DIR, OUTPUT_TRAIN_DIR, "Cutting Train Masks")
process_cutting(INPUT_INF_CSV, INF_MASK_DIR, OUTPUT_INF_DIR, "Cutting Inference Masks")

print("Cutting complete. Check 'cleaned_dataset/cut/' for results.")

Cutting Inference Masks: 100%|██████████| 4384/4384 [00:06<00:00, 711.37it/s]

Cutting complete. Check 'cleaned_dataset/cut/' for results.


In [10]:
import os
import pandas as pd
import numpy as np
import cv2
from scipy.interpolate import make_interp_spline

# --- Configuration ---
MASK_DIR = r'cleaned_dataset\split\train_masks'
CSV_INPUT = r'cleaned_dataset\new_csv\train_metadata_with_cutoffs.csv'
CSV_OUTPUT = r'cleaned_dataset\new_csv\train_full.csv'
RECON_OUT_DIR = r'cleaned_dataset\recon\train'

os.makedirs(RECON_OUT_DIR, exist_ok=True)

def parse_coord(coord_str):
    if pd.isna(coord_str) or coord_str == "":
        return None
    try:
        cleaned = str(coord_str).replace('[', '').replace(']', '').strip()
        return np.array([int(float(c)) for c in cleaned.split(',')])
    except:
        return None

def process_dataset():
    df = pd.read_csv(CSV_INPUT)
    
    for idx, row in df.iterrows():
        img_name = row['image_path']
        img_path = os.path.join(MASK_DIR, img_name)
        if not os.path.exists(img_path):
            continue

        # 1. Load + preprocess
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        cnt_sq = cnt.squeeze()

        l_cutoff = parse_coord(row['cutoff_l'])
        r_cutoff = parse_coord(row['cutoff_r'])
        if l_cutoff is None or r_cutoff is None:
            continue

        L_mm, W_mm = row['length_mm'], row['width_mm']

        x_min, x_max = np.min(cnt_sq[:, 0]), np.max(cnt_sq[:, 0])
        y_min, y_max = np.min(cnt_sq[:, 1]), np.max(cnt_sq[:, 1])

        x_center = x_min + (x_max - x_min) // 2
        y_bottom_target = y_min + int((x_max - x_min) * (L_mm / W_mm))

        # 2. Split contour
        left_half = cnt_sq[cnt_sq[:, 0] < x_center]
        right_half = cnt_sq[cnt_sq[:, 0] >= x_center]

        if len(left_half) == 0 or len(right_half) == 0:
            continue

        # guide points (slightly above cutoff)
        l_guide = left_half[np.argmin(np.abs(left_half[:, 1] - (l_cutoff[1] - 15)))]
        r_guide = right_half[np.argmin(np.abs(right_half[:, 1] - (r_cutoff[1] - 15)))]

        anchors = np.array([
            l_guide,
            l_cutoff,
            [x_center, y_bottom_target],
            r_cutoff,
            r_guide
        ])

        # 3. Spline (chord-length parameterization)
        distances = np.linalg.norm(np.diff(anchors, axis=0), axis=1)
        u = np.insert(np.cumsum(distances), 0, 0)

        spline_x = make_interp_spline(u, anchors[:, 0], k=3)
        spline_y = make_interp_spline(u, anchors[:, 1], k=3)

        u_fine = np.linspace(u[1], u[3], 100)
        curve_pts = np.column_stack((spline_x(u_fine), spline_y(u_fine))).astype(np.int32)

        # 4. Visualization (WHITE bg, BLACK lines)
        h, w = img.shape[:2]
        canvas = np.ones((h, w, 3), dtype=np.uint8) * 255  # white

        # clipping mask (keep only upper contour)
        if (r_cutoff[0] - l_cutoff[0]) != 0:
            m = (r_cutoff[1] - l_cutoff[1]) / (r_cutoff[0] - l_cutoff[0])
        else:
            m = 0
        b = l_cutoff[1] - m * l_cutoff[0]

        mask = np.zeros((h, w), dtype=np.uint8)
        clip_poly = np.array([
            [0, 0],
            [w, 0],
            [w, int(m * w + b)],
            [0, int(m * 0 + b)]
        ], dtype=np.int32)

        cv2.fillPoly(mask, [clip_poly], 255)

        # draw contour on temp layer
        temp = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(temp, [cnt], -1, 255, 1)

        # apply clipping
        upper_only = cv2.bitwise_and(temp, temp, mask=mask)

        # paint onto white canvas
        canvas[upper_only > 0] = [0, 0, 0]

        # draw spline (black)
        cv2.polylines(
            canvas,
            [curve_pts.reshape((-1, 1, 2))],
            isClosed=False,
            color=(0, 0, 0),
            thickness=1
        )

        # 5. Save outputs
        df.at[idx, 'end_pt'] = f"{x_center},{y_bottom_target}"
        df.at[idx, 'curve'] = "|".join([f"{p[0]},{p[1]}" for p in curve_pts])

        cv2.imwrite(os.path.join(RECON_OUT_DIR, img_name), canvas)

    df.to_csv(CSV_OUTPUT, index=False)
    print("Done. White background + black outlines.")

process_dataset()

C:\Users\Dell\AppData\Local\Temp\ipykernel_16784\657368552.py:128: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '258,461' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'end_pt'] = f"{x_center},{y_bottom_target}"
C:\Users\Dell\AppData\Local\Temp\ipykernel_16784\657368552.py:129: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '168,418|169,419|171,420|172,422|174,423|175,424|177,426|179,427|180,428|182,430|184,431|185,432|187,434|189,435|191,436|192,437|194,438|196,440|198,441|200,442|201,443|203,444|205,445|207,446|209,447|211,448|213,449|214,450|216,451|218,452|220,452|222,453|224,454|226,455|228,455|230,456|232,456|234,457|236,458|238,458|240,458|242,459|244,459|245,460|247,460|249,460|251,460|253,460|255,460|257,460|259,460|261,460|263,460|265,460|267,460|269,460|271,4

Done. White background + black outlines.
